# ACT로 모방학습하기

## 기본설정

In [1]:
import time
from types import SimpleNamespace

In [2]:
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

from lerobot.robots.so_follower import SO101Follower, SO101FollowerConfig
from lerobot.teleoperators.so_leader import SO101Leader, SO101LeaderConfig

from lerobot.processor import make_default_processors

In [3]:
dataset = LeRobotDataset("lerobot/pusht")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
teleop_action_processor, robot_action_processor, robot_observation_processor = make_default_processors()

In [5]:
robot_config = SO101FollowerConfig(
    port="/dev/ttyACM0",
    id="lhwdev_follower3",
)

leader_config = SO101LeaderConfig(
    port="/dev/ttyACM1",
    id="lhwdev_leader",
)

robot = SO101Follower(robot_config)
leader = SO101Leader(leader_config)

robot.connect()
leader.connect()

RuntimeError: FeetechMotorsBus motor check failed on port '/dev/ttyACM1':

Missing motor IDs:
  - 1 (expected model: 777)
  - 2 (expected model: 777)
  - 3 (expected model: 777)
  - 4 (expected model: 777)
  - 5 (expected model: 777)
  - 6 (expected model: 777)

Full expected motor list (id: model_number):
{1: 777, 2: 777, 3: 777, 4: 777, 5: 777, 6: 777}

Full found motor list (id: model_number):
{}

In [ ]:
import cv2

cam = cv2.VideoCapture(2)

## teleoperate

In [7]:
from IPython.display import display, Image
import cv2

_text_handle = None
_img_handle = None

def show_image(fps_text=None):
    global _text_handle, _img_handle
    
    ret, frame = cam.read()
    if not ret:
        return
        
    _, jpeg = cv2.imencode('.jpg', frame)
    jpeg_bytes = jpeg.tobytes()
    img_obj = Image(data=jpeg_bytes)
    
    if fps_text is not None:
        if _text_handle is None:
            _text_handle = display(fps_text, display_id=True)
        else:
            _text_handle.update(fps_text)
            
    if _img_handle is None:
        _img_handle = display(img_obj, display_id=True)
    else:
        _img_handle.update(img_obj)


In [8]:
def sync_to_robot():
    obs = robot.get_observation()
    raw_action = leader.get_action()
    teleop_action = teleop_action_processor((raw_action, obs))
    robot_action = robot_action_processor((teleop_action, obs))

    robot.send_action(robot_action)

In [ ]:
from IPython.display import display, clear_output

start_time = time.perf_counter()
iterations = 0

# Reset display handles so a new display is created in this cell's output
_text_handle = None
_img_handle = None

while True:
    sync_to_robot()
    iterations += 1

    if iterations % 100 == 0:
        elipsed = time.perf_counter() - start_time
        fps_text = f"fps={iterations / elipsed:.3f} elipsed={elipsed:.2f}"
        show_image(fps_text)


## 데이터셋 수집

In [10]:
import importlib
from record_interactive import record_interactive

record_interactive(dataset, robot, leader, cam)

HTML(value="\n    <style>\n    .studio-header {\n        background: linear-gradient(135deg, hsl(260, 80%, 40%…

Error in studio loop:
Traceback (most recent call last):
  File "/home/lhwdev/csi-agent/lerobot/lhwdev/record_interactive.py", line 570, in main_loop
    obs = robot.get_observation()
          ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lhwdev/csi-agent/lerobot/src/lerobot/utils/decorators.py", line 29, in wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lhwdev/csi-agent/lerobot/src/lerobot/robots/so_follower/so_follower.py", line 181, in get_observation
    obs_dict = self.bus.sync_read("Present_Position")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lhwdev/csi-agent/lerobot/src/lerobot/utils/decorators.py", line 29, in wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lhwdev/csi-agent/lerobot/src/lerobot/motors/motors_bus.py", line 1157, in sync_read
    raw_ids_values, _ = self._sync_read(
                        ^^^^^^^^^^^^^^^^
  File "/home/lhwdev/csi-agent/lerob

## 이제 학습!

In [ ]:
dataset_metadata = LeRobotDatasetMetadata(
    "lhwdev/move_cube",
    root="/home/lhwdev/csi-agent/lerobot/lhwdev/records/move_cube"
)

train_params = SimpleNamespace(
    output_directory = "train/move_cube",
    batch_size = 8,

    training_steps = 1000,
    save_freq = 200,
    log_freq = 100,
)

from train import train_simple
train_simple(train_params, dataset_metadata)

Using device: xpu
No existing checkpoint found. Starting training from scratch...
Staging training from step 0 to 1000...


Training:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved checkpoint to train/move_cube/checkpoints/000200


In [19]:
!lerobot-rollout \
    --strategy.type=base \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=lhwdev_follower3 \
    --robot.cameras='{front: {type: opencv, index_or_path: 2, width: 640, height: 480, fps: 30}}' \
    --policy.path='./train/move_cube/checkpoints/last/pretrained_model' \
    --task='Pick the blue cube, and put in red dish.'

/bin/bash: /home/lhwdev/miniconda3/envs/lerobot/lib/libtinfo.so.6: no version information available (required by /bin/bash)
INFO 2026-06-26 14:19:23 _rollout.py:210 Building rollout context...
INFO 2026-06-26 14:19:23 /context.py:177 Loading policy from './train/move_cube/checkpoints/last/pretrained_model'...
Loading weights from local directory
INFO 2026-06-26 14:19:24 /context.py:209 Policy loaded: type=act, device=xpu
INFO 2026-06-26 14:19:24 /context.py:236 Connecting robot (so100_follower)...
INFO 2026-06-26 14:19:25 a_opencv.py:179 OpenCVCamera(2) connected.
INFO 2026-06-26 14:19:25 follower.py:105 lhwdev_follower3 SOFollower connected.
INFO 2026-06-26 14:19:25 /context.py:239 Robot connected: so_follower
INFO 2026-06-26 14:19:25 /context.py:244 Captured initial robot position (6 keys)
INFO 2026-06-26 14:19:25 /context.py:408 Creating inference engine (type=sync)...
INFO 2026-06-26 14:19:25 /factory.py:100 Creating inference engine: sync
INFO 2026-06-26 14:19:25 ence/sync.py:76 S

In [ ]:
!python ../src/lerobot/scripts/control_robot.py \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=lhwdev_follower3 \
    --control.fps=30 \
    --control.policy.path='train/pick_umbrella3'

python: can't open file '/home/lhwdev/csi-agent/lerobot/lhwdev/../scripts/control_robot.py': [Errno 2] No such file or directory


In [4]:
!lerobot-train \
    --dataset.repo_id='lhwdev/pick_umbrella' \
    --dataset.root='/home/lhwdev/csi-agent/lerobot/lhwdev/records/pick_umbrella3' \
    --output_dir=train/pick_umbrella3_smolvla \
    --policy.path=lerobot/smolvla_base \
    --policy.device=cuda \
    --policy.push_to_hub=false \
    --policy.output_features=null \
    --policy.input_features=null \
    --policy.optimizer_lr=1e-3 \
    --policy.scheduler_decay_lr=1e-4 \
    --peft.method_type=LORA \
    --peft.r=64 \
    --steps=100 \
    --batch_size=32

config.json: 100%|████████████████████████| 2.30k/2.30k [00:00<00:00, 3.38MB/s]
INFO 2026-06-25 17:26:13 ot_train.py:221 {'batch_size': 32,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                     